**IMPORTING MODULES**

In [1]:
import moviepy as mp
from pydub import AudioSegment
import os
#moviepy and pydub are used for audio/video handling.
#Uses pydub to convert the MP3 file into a WAV file (needed for speech recognition).
#os is for system-related operations.

**CONVERTING VIDEO TO AUDIO**

In [3]:
import yt_dlp
#yt_dlp is used to download media from YouTube.
# Replace this path with the actual path to your ffmpeg 'bin' folder
ffmpeg_path = r"C:\Users\sivak\OneDrive\Desktop\ffmpeg-7.1.1-essentials_build\bin"
os.environ["PATH"] += os.pathsep + ffmpeg_path
video_url = input("Enter the YouTube video URL: ")
#Adds your local FFmpeg path to the environment, so Python can use it to process media files.
ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': 'downloaded_audio.%(ext)s',
    'ffmpeg_location': ffmpeg_path,  # This tells yt_dlp exactly where ffmpeg is
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',#Download best audio.
        'preferredcodec': 'mp3',#Save it as downloaded_audio.mp3
        'preferredquality': '192',#Use FFmpeg to convert it to MP3 with quality 192 kbps
    }],
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    print("Downloading and extracting audio...")
    ydl.download([video_url])

print("Audio saved as downloaded_audio.mp3")

Enter the YouTube video URL:  https://www.youtube.com/watch?v=d0Anl3tIKaA


[youtube] Extracting URL: https://www.youtube.com/watch?v=d0Anl3tIKaA
[youtube] d0Anl3tIKaA: Downloading webpage
[youtube] d0Anl3tIKaA: Downloading tv client config
[youtube] d0Anl3tIKaA: Downloading player 8102da6c-main
[youtube] d0Anl3tIKaA: Downloading tv player API JSON
[youtube] d0Anl3tIKaA: Downloading ios player API JSON
[youtube] d0Anl3tIKaA: Downloading m3u8 information
[info] d0Anl3tIKaA: Downloading 1 format(s): 251
[download] Destination: downloaded_audio.webm
[download] 100% of    3.60MiB in 00:00:00 at 11.34MiB/s  
[ExtractAudio] Destination: downloaded_audio.mp3
Deleting original file downloaded_audio.webm (pass -k to keep)
Audio saved as downloaded_audio.mp3


**AUDIO TO TEXT**

In [5]:
import speech_recognition as sr
mp3_path = "downloaded_audio.mp3"
wav_path = "converted_audio.wav"#It stores raw audio, so there's no quality loss.

print("Converting MP3 to WAV...")
audio = AudioSegment.from_mp3(mp3_path)
audio.export(wav_path, format="wav")
print("Conversion complete. Saved as converted_audio.wav")

# --- Step 3: Transcribe audio in chunks ---
recognizer = sr.Recognizer() #Defines chunk size as 60 seconds (to process large audio piece-by-piece).
chunk_length = 60 * 1000  # 60 seconds in milliseconds

print("Splitting and transcribing audio...")
audio = AudioSegment.from_wav(wav_path)
chunks = len(audio) // chunk_length + 1

full_transcript = []

for i in range(chunks):
    chunk = audio[i*chunk_length : (i+1)*chunk_length]
    chunk_name = f"chunk{i}.wav"
    chunk.export(chunk_name, format="wav")
    
    with sr.AudioFile(chunk_name) as source:
        audio_data = recognizer.record(source)
        try:
            text = recognizer.recognize_google(audio_data)
            print(f"Chunk {i+1} Transcript:\n{text}\n")
            full_transcript.append(text)
        except sr.UnknownValueError:
            print(f"Chunk {i+1}: Could not understand audio.")
        except sr.RequestError as e:
            print(f"Chunk {i+1}: API request error - {e}")

# Optional: Save full transcript
with open("full_transcript.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(full_transcript))

print("Transcription complete. Saved to full_transcript.txt")

Converting MP3 to WAV...
Conversion complete. Saved as converted_audio.wav
Splitting and transcribing audio...
Chunk 1 Transcript:
the Indus water Treaty signed in 1960 is often held as one of the most successful water sharing agreements in history brokered by the World bank between India and Pakistan it has survived was terrorism and decades of hostility however the Pahalgam terror attack on April 22nd has once again post this historic Treaty in determinant waters how can I explain

Chunk 2 Transcript:
fraud Kiya Hamare Sath aur Musalman Karte Rahe Hain Kyunki vah Jo ling ke naam se jo flood Canal se unko jo hai vah 12 mahine sal ke Jo Hai vah Khula rakhte hain aur Hamare Pani Ki Chori Bhi Karte Hain after the partition of 1947 India and Pakistan inherited a shared River system The Mighty Indus and its five tributaries the Jhelum Chenab Ravi Beas and Sutlej with water crucial for agriculture survival and economic development disputes erupted almost immediately realising the risk of an

**TEXT TO MULTI LANGUAGE**

In [7]:
from googletrans import Translator

lang_input = input("Enter languages to translate into (e.g., Tamil, Hindi, French): ")
language_names = [name.strip() for name in lang_input.split(",")]

# 🔠 Language name to code map
lang_map = {
    'Tamil': 'ta',
    'Hindi': 'hi',
    'Telugu': 'te',
    'French': 'fr',
    'Spanish': 'es'
}
selected_langs = {lang: lang_map[lang] for lang in language_names if lang in lang_map}

# 🌐 Translate entire passage
translator = Translator()
transcript_text = "\n\n".join(full_transcript)

print("\n--- Full Translations ---\n")
for lang_name, lang_code in selected_langs.items():
    try:
        translated = translator.translate(transcript_text, dest=lang_code)
        print(f"{lang_name} Translation:\n{translated.text}\n{'-'*50}")
    except Exception as e:
        print(f"❌ Error translating to {lang_name}: {e}")


Enter languages to translate into (e.g., Tamil, Hindi, French):  Telugu



--- Full Translations ---

Telugu Translation:
1960 లో సంతకం చేసిన సింధు నీటి ఒప్పందం తరచుగా భారతదేశం మరియు పాకిస్తాన్ మధ్య ప్రపంచ బ్యాంక్ బ్రోకర్ చేసిన చరిత్రలో అత్యంత విజయవంతమైన నీటి భాగస్వామ్య ఒప్పందాలలో ఒకటిగా ఉంది, ఇది ఉగ్రవాదం మరియు దశాబ్దాల శత్రుత్వం, అయితే ఏప్రిల్ 22 న పహల్గామ్ టెర్రర్ దాడి మరోసారి ఈ చారిత్రక ఒప్పందాన్ని నిర్ణయాత్మక నీటిలో పోస్ట్ చేసింది, నేను ఎలా వివరించగలను

మోసం కియా హమారే సత్ ur ర్ ముసల్మాన్ కార్టే రహే హైన్ క్యూంకి వా జో లింగ్ కే నామ్ సే జో ఫ్లడ్ కెనాల్ కానాల్ సే ఉన్కో జో హై వాహ్ 12 మహేన్ సాల్ కే జో హై వాహ్ ఖులా రాఖే రాఖే హైన్ రఖే హైన్ ur ర్ హరాన్ పాణికి చోరి భి కర్టే హైన్ 1947 ఇండియస్ ఇండస్ హైన్వ్యవసాయ మనుగడ మరియు ఆర్థిక అభివృద్ధి వివాదాలకు కీలకమైన నీటితో జీలం చెనాబ్ రవి బీస్ మరియు సుట్లెజ్ ఈసారి మరో పదం యొక్క ప్రమాదాన్ని గ్రహించాయి, ఈసారి ప్రపంచ బ్యాంకు మధ్యవర్తిత్వం ఉన్న రెండు దేశాలు 1960 నదులు మరియు రవిలలో కరాచీలో సింధు నీటి ఒప్పందంపై సంతకం చేశాయి

భారతదేశానికి కేటాయించగా, పశ్చిమ నదులు సింధు జీలం మరియు చెనాబ్ పాకిస్తాన్‌కు కేటాయించిన సింధు నీటి ఒప్పందా